# Word2Vec using Gensim

In [ ]:
import subprocess
import sys
import os
import numpy as np
from scipy.stats import spearmanr, pearsonr

# ---------------- INSTALL ----------------
required = ["gensim", "numpy", "scipy", "transformers", "torch"]
for lib in required:
    try:
        __import__(lib)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])

from gensim.models import FastText, KeyedVectors
from transformers import AutoTokenizer, AutoModel
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- TEST DATA ----------------
isi_test_pairs = [
    ('inkosi', 'imeya', 8.45),
    ('imali', 'isikweletu', 7.12),
    ('uhulumeni', 'umasipala', 8.90),
    ('inkohlakalo', 'icala', 7.50),
    ('isikole', 'inyuvesi', 8.20),
    ('umfundi', 'uthisha', 7.65),
    ('itekisi', 'ubudokotela', 1.15),
    ('umculo', 'isifo', 0.90),
]

vocab = list(set([w for pair in isi_test_pairs for w in pair[:2]]))

# ---------------- COSINE ----------------
def cosine(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# ---------------- RETRIEVAL METRICS ----------------
def compute_retrieval_metrics(embedding_func):
    P1, P5, P10 = [], [], []
    AP_scores = []

    for w1, w2, score in isi_test_pairs:
        if score < 7:  # treat >=7 as synonym
            continue

        sims = []
        for candidate in vocab:
            if candidate == w1:
                continue
            try:
                sim = cosine(embedding_func(w1), embedding_func(candidate))
                sims.append((candidate, sim))
            except:
                continue

        sims.sort(key=lambda x: x[1], reverse=True)
        ranked = [x[0] for x in sims]

        # Precision@k
        P1.append(1 if w2 == ranked[:1][0] else 0)
        P5.append(1 if w2 in ranked[:5] else 0)
        P10.append(1 if w2 in ranked[:10] else 0)

        # Average Precision
        hits = 0
        precision_sum = 0
        for i, candidate in enumerate(ranked):
            if candidate == w2:
                hits += 1
                precision_sum += hits / (i+1)
        AP_scores.append(precision_sum / hits if hits > 0 else 0)

    return (
        np.mean(P1),
        np.mean(P5),
        np.mean(P10),
        np.mean(AP_scores)
    )

# ---------------- LOCAL FASTTEXT ----------------
sentences = [
    ["inkosi","imeya","umbuso"],
    ["imali","isikweletu","ibhange"],
    ["isikole","inyuvesi","umfundi"],
    ["umculo","ingoma","umculi"],
    ["itekisi","imoto","umgwaqo"],
]

print("\nTraining Local FastText...")
local_ft = FastText(
    sentences=sentences,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1,
    epochs=100,
    min_n=3,
    max_n=6
)

# ---------------- PRETRAINED FASTTEXT CC ----------------
if not os.path.exists("cc.zu.300.vec"):
    print("Download cc.zu.300.vec from:")
    print("https://fasttext.cc/docs/en/crawl-vectors.html")
    sys.exit()

print("Loading fastText CC...")
ft_cc = KeyedVectors.load_word2vec_format("cc.zu.300.vec")

# ---------------- LOAD mBERT ----------------
print("Loading mBERT...")
mbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
mbert_model = AutoModel.from_pretrained("bert-base-multilingual-cased").to(device)

def mbert_embedding(word):
    inputs = mbert_tokenizer(word, return_tensors="pt").to(device)
    outputs = mbert_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()[0]

# ---------------- LOAD XLM-R ----------------
print("Loading XLM-R...")
xlmr_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
xlmr_model = AutoModel.from_pretrained("xlm-roberta-base").to(device)

def xlmr_embedding(word):
    inputs = xlmr_tokenizer(word, return_tensors="pt").to(device)
    outputs = xlmr_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().cpu().numpy()[0]

# ---------------- EVALUATION ----------------
def evaluate(model_name, embedding_func):
    human = []
    cosine_scores = []

    for w1, w2, score in isi_test_pairs:
        try:
            v1 = embedding_func(w1)
            v2 = embedding_func(w2)
            cos = cosine(v1, v2)

            human.append(score)
            cosine_scores.append(cos)
        except:
            continue

    rho, _ = spearmanr(human, cosine_scores)
    pear, _ = pearsonr(human, cosine_scores)
    P1, P5, P10, MAP = compute_retrieval_metrics(embedding_func)

    print(f"\n{model_name}")
    print(f"Spearman: {rho:.4f}")
    print(f"Pearson : {pear:.4f}")
    print(f"P@1     : {P1:.4f}")
    print(f"P@5     : {P5:.4f}")
    print(f"P@10    : {P10:.4f}")
    print(f"MAP     : {MAP:.4f}")

    return rho, pear, P1, P5, P10, MAP

print("\nEvaluating Models...")
print("="*50)

results = {}

results["Local FastText"] = evaluate(
    "Local FastText",
    lambda w: local_ft.wv[w]
)

results["fastText CC"] = evaluate(
    "fastText CC",
    lambda w: ft_cc[w]
)

results["mBERT"] = evaluate(
    "mBERT",
    lambda w: mbert_embedding(w)
)

results["XLM-R"] = evaluate(
    "XLM-R",
    lambda w: xlmr_embedding(w)
)

# ---------------- FINAL TABLE ----------------
print("\n" + "="*70)
print("FINAL COMPARISON TABLE")
print("="*70)
print(f"{'Model':<20}{'Spearman':<10}{'Pearson':<10}{'P@1':<8}{'P@5':<8}{'P@10':<8}{'MAP':<8}")

for model in results:
    r = results[model]
    print(f"{model:<20}{r[0]:<10.4f}{r[1]:<10.4f}{r[2]:<8.4f}{r[3]:<8.4f}{r[4]:<8.4f}{r[5]:<8.4f}")

print("\nDone ✅ Publication-level comparison complete.")

C:\Users\USER-PC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Training Local FastText...
Download cc.zu.300.vec from:
https://fasttext.cc/docs/en/crawl-vectors.html


SystemExit: 

C:\Users\USER-PC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
